In [1]:
import os, re, glob, itertools
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    balanced_accuracy_score, f1_score, recall_score, average_precision_score
)

# Group-aware CV (newer sklearn)
try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_SGK = True
except Exception:
    HAS_SGK = False


# ----------------------------
# Inputs / labels  (LOCAL paths)
# ----------------------------
PROJECT_DIR = "/Users/leoqian/mdanderson/EEGOutcomePrediction"
DATA_DIR    = f"{PROJECT_DIR}/processeddata"

df = pd.read_excel(f"{DATA_DIR}/Randomization factors and Primary outcome.xlsx")
diff = df[df['Event Name'].isin(['T1', 'T2'])].pivot_table(
    index='Patient number', columns='Event Name', values='Pain Unpleasantness'
).dropna()
diff['Diff'] = diff['T2'] - diff['T1']

# Local processed-data dir holds CIPN3*.xlsx (one file per patient).
output_dir = f"{DATA_DIR}/"
median_threshold = -2.00

# Z_FFT_abs_bandpower_uV2
sheet_name = "Z_FFT_abs_bandpower_uV2"

# If you still want your old per-sheet defaults, keep them here
# (but since you asked explicitly, we'll use selected_columns above)
# if sheet_name == "Z_FFT_Coherence":
#     selected_columns = ['Delta', 'Theta', 'Alpha', 'Beta', 'HighBeta', 'Alpha1', 'Alpha2', 'Beta1', 'Beta2', 'Beta3']
if sheet_name == "Z_FFT_abs_bandpower_uV2":
    selected_columns = ['Delta', 'Theta', 'Alpha', 'Beta', 'HighBeta', 'Alpha1', 'Alpha2', 'Beta1', 'Beta2', 'Beta3']
# elif sheet_name == "Z_PeakFreq_Hz":
#     selected_columns = ['Theta', 'Alpha', 'Beta', 'HighBeta']
# elif sheet_name == "Z_FFT_PhaseLag_PLI":
#     selected_columns = ['Delta', 'Theta', 'Alpha', 'Beta', 'HighBeta', 'Alpha1', 'Alpha2', 'Beta1', 'Beta2', 'Beta3']
# else:
#     selected_columns = None

# LR hyperparameter grid
solvers = ["liblinear", "saga"]
penalties = ["l1", "l2"]
C_values = [1e-3, 1e-2, 1e-1, 1, 10, 100]

min_patients_required = 20

# CV config
outer_n_splits = 5
outer_n_repeats = 2
inner_n_splits = 5


# ----------------------------
# Utilities
# ----------------------------
def parse_patient_id_from_filename(xlsx_path: str):
    base = os.path.basename(xlsx_path)
    m = re.search(r"CIPN3(\d{3})", base)
    return int(m.group(1)) if m else None

def build_samples(output_dir: str):
    files = sorted(glob.glob(os.path.join(output_dir, "CIPN3*.xlsx")))
    rows = []
    for f in files:
        pid = parse_patient_id_from_filename(f)
        if pid is None:
            continue
        rows.append((pid, f))
    return rows

def all_nonempty_subsets(cols):
    cols = list(cols)
    out = []
    for r in range(1, len(cols) + 1):
        out.extend(list(itertools.combinations(cols, r)))
    return out

def feats_from_single_band_column(df_band: pd.DataFrame) -> np.ndarray:
    """
    df_band: a 1-col dataframe for one band.
    Produces 5 features per band:
      [col_mean, col_std, global_mean, global_std, global_median]
    """
    A = df_band.values.astype(float)
    A = np.nan_to_num(A, nan=0.0, posinf=0.0, neginf=0.0)

    col_mean = A.mean(axis=0)[0]
    col_std  = A.std(axis=0)[0]
    g_mean   = A.mean()
    g_std    = A.std()
    g_med    = float(np.median(A))

    return np.array([col_mean, col_std, g_mean, g_std, g_med], dtype=float)

def load_bandwise_cache_for_file(xlsx_path: str, sheet_name: str, bands: list):
    """
    Returns dict band->(5,) feature vector for that band (if present in sheet).
    """
    df_sheet = pd.read_excel(xlsx_path, sheet_name=sheet_name, index_col=0)
    out = {}
    for b in bands:
        if b in df_sheet.columns:
            out[b] = feats_from_single_band_column(df_sheet[[b]])
    return out

def build_X_from_subset(cache_list, subset):
    """
    cache_list: list of dicts (per sample) band->(5,)
    subset: tuple/list of band names
    Returns X (n_samples, 5*len(subset)) or None if any sample missing a band.
    """
    X_rows = []
    for d in cache_list:
        # require all bands in subset
        if any(b not in d for b in subset):
            return None
        row = np.concatenate([d[b] for b in subset], axis=0)
        X_rows.append(row)
    return np.vstack(X_rows)


def make_outer_splits(X, y, groups, n_splits, n_repeats, base_seed=42):
    """
    Returns list of (train_idx, test_idx) with grouping by patient.
    """
    all_splits = []
    for r in range(n_repeats):
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=base_seed + r)
        for tr, te in cv.split(X, y, groups=groups):
            all_splits.append((tr, te))
    return all_splits

def mean_std(x):
    x = np.asarray(x, dtype=float)
    return float(x.mean()), float(x.std())


# ----------------------------
# Build sample list (pid, file), keep only labeled pids
# ----------------------------
if not HAS_SGK:
    raise RuntimeError(
        "Your sklearn does not have StratifiedGroupKFold. "
        "Upgrade sklearn or ask me for a patient-collapsed fallback."
    )

rows = build_samples(output_dir)
rows = [(pid, f) for (pid, f) in rows if pid in diff.index]

if len(rows) == 0:
    raise RuntimeError("No samples found after filtering for labeled patients.")

print(f"Found {len(rows)} files with labels. Building patient-level labels...")

# patient-level label
unique_pids = sorted(set([pid for pid, _ in rows]))
y_patient = (diff.loc[unique_pids, "Diff"] >= median_threshold).astype(int)
pid_to_y = y_patient.to_dict()

# sample-wise y/groups/files
y = np.asarray([int(pid_to_y[pid]) for pid, _ in rows], dtype=int)
groups = np.asarray([pid for pid, _ in rows], dtype=int)
kept_files = [f for _, f in rows]

print("Unique patients:", len(np.unique(groups)))
print(f"Threshold: {median_threshold} | Neg: {(y==0).sum()} Pos: {(y==1).sum()} | AP baseline: {y.mean():.3f}")

if len(np.unique(groups)) < min_patients_required:
    raise RuntimeError(f"Too few unique patients: {len(np.unique(groups))} (< {min_patients_required})")


# ----------------------------
# Pre-load band-wise feature caches (so subset search is fast)
# ----------------------------
print(f"\nLoading band-wise features from sheet='{sheet_name}' for bands={selected_columns}")
band_cache_all = []
ok_count = 0
for i, f in enumerate(kept_files, 1):
    try:
        d = load_bandwise_cache_for_file(f, sheet_name, selected_columns)
        band_cache_all.append(d)
        ok_count += 1
    except Exception as e:
        band_cache_all.append({})
        print(f"  [WARN] failed to read {os.path.basename(f)}: {e}")

print(f"Loaded caches for {ok_count}/{len(kept_files)} files.")
print("Example cache keys for first sample:", list(band_cache_all[0].keys())[:10])

# Quick sanity: at least one band present somewhere
if all(len(d) == 0 for d in band_cache_all):
    raise RuntimeError(f"No band columns found in sheet '{sheet_name}' across all files.")


# ----------------------------
# Model + Grid
# ----------------------------
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        class_weight="balanced",
        max_iter=50000,
        tol=1e-3,
        random_state=42
    ))
])

param_grid = []
for solver in solvers:
    for penalty in penalties:
        grid_entry = {
            "clf__solver": [solver],
            "clf__penalty": [penalty],
            "clf__C": C_values
        }
        if penalty == "l1":
            grid_entry["clf__warm_start"] = [True]
        param_grid.append(grid_entry)


# ----------------------------
# Outer splits (patient-wise)
# ----------------------------
# We need a dummy X for split generation; use 1D placeholder (len = n_samples)
X_dummy = np.zeros((len(y), 1), dtype=float)

outer_splits = make_outer_splits(X_dummy, y, groups, outer_n_splits, outer_n_repeats, base_seed=42)
print(f"\nOuter CV: {outer_n_splits} splits x {outer_n_repeats} repeats = {len(outer_splits)} folds total.")

candidate_subsets = all_nonempty_subsets(selected_columns)
print(f"Band-subset search space: {len(candidate_subsets)} subsets (non-empty).")


# ----------------------------
# Nested CV with band-subset selection inside each outer fold
# ----------------------------
test_metrics = {"bal_acc": [], "f1": [], "recall_pos": [], "recall_neg": [], "ap": []}
train_metrics = {"bal_acc": [], "f1": [], "recall_pos": [], "recall_neg": [], "ap": []}
chosen_subsets = []

for fold_i, (tr, te) in enumerate(outer_splits, 1):
    g_tr, g_te = groups[tr], groups[te]

    # sanity: no patient overlap
    if len(set(g_tr) & set(g_te)) != 0:
        raise RuntimeError("Leakage detected: a patient appears in both train and test in a fold.")

    print("\n" + "="*90)
    print(f"[Outer fold {fold_i}/{len(outer_splits)}] "
          f"train_samples={len(tr)} test_samples={len(te)} | "
          f"train_patients={len(np.unique(g_tr))} test_patients={len(np.unique(g_te))}")
    print("Searching best band subset via inner CV...")

    inner_cv = StratifiedGroupKFold(n_splits=inner_n_splits, shuffle=True, random_state=1)

    best_subset = None
    best_inner_score = -1
    best_gs = None
    best_X_tr = None
    best_X_te = None

    # pre-slice caches for speed
    cache_tr = [band_cache_all[i] for i in tr]
    cache_te = [band_cache_all[i] for i in te]

    # progress print every few subsets
    progress_every = max(1, len(candidate_subsets)//10)

    for si, subset in enumerate(candidate_subsets, 1):
        X_tr_sub = build_X_from_subset(cache_tr, subset)
        if X_tr_sub is None:
            continue
        X_te_sub = build_X_from_subset(cache_te, subset)
        if X_te_sub is None:
            continue

        # GridSearch on LR params with inner CV
        gs = GridSearchCV(
            pipe,
            param_grid=param_grid,
            scoring="balanced_accuracy",
            cv=inner_cv,
            n_jobs=-1,
            refit=True
        )
        gs.fit(X_tr_sub, y[tr], groups=g_tr)

        score = gs.best_score_
        if score > best_inner_score:
            best_inner_score = score
            best_subset = subset
            best_gs = gs
            best_X_tr = X_tr_sub
            best_X_te = X_te_sub

        if (si % progress_every) == 0:
            print(f"  progress: {si}/{len(candidate_subsets)} subsets tried | current best "
                  f"{best_subset} score={best_inner_score:.3f}")

    if best_subset is None:
        raise RuntimeError("No valid subset found (likely missing bands across samples).")

    chosen_subsets.append(best_subset)
    print(f"\nBest subset for this outer fold: {best_subset} | best inner bal_acc={best_inner_score:.3f}")
    print("Best LR params:", best_gs.best_params_)

    best_est = best_gs.best_estimator_

    # probabilities for AP
    p_tr = best_est.predict_proba(best_X_tr)[:, 1]
    p_te = best_est.predict_proba(best_X_te)[:, 1]

    # Tune threshold on outer-train set to maximize balanced accuracy
    ths = np.linspace(0.05, 0.95, 181)
    best_th, best_ba = 0.5, -1
    for t in ths:
        yhat = (p_tr >= t).astype(int)
        ba = balanced_accuracy_score(y[tr], yhat)
        if ba > best_ba:
            best_ba, best_th = ba, t

    yhat_tr = (p_tr >= best_th).astype(int)
    yhat_te = (p_te >= best_th).astype(int)

    # train metrics
    train_bal_acc = balanced_accuracy_score(y[tr], yhat_tr)
    train_metrics["bal_acc"].append(train_bal_acc)
    train_metrics["f1"].append(f1_score(y[tr], yhat_tr, zero_division=0))
    train_metrics["recall_pos"].append(recall_score(y[tr], yhat_tr, pos_label=1, zero_division=0))
    train_metrics["recall_neg"].append(recall_score(y[tr], yhat_tr, pos_label=0, zero_division=0))
    train_metrics["ap"].append(average_precision_score(y[tr], p_tr))

    # test metrics
    test_bal_acc = balanced_accuracy_score(y[te], yhat_te)
    test_metrics["bal_acc"].append(test_bal_acc)
    test_metrics["f1"].append(f1_score(y[te], yhat_te, zero_division=0))
    test_metrics["recall_pos"].append(recall_score(y[te], yhat_te, pos_label=1, zero_division=0))
    test_metrics["recall_neg"].append(recall_score(y[te], yhat_te, pos_label=0, zero_division=0))
    test_metrics["ap"].append(average_precision_score(y[te], p_te))

    print(f"Threshold tuned on train: best_th={best_th:.3f} (train_bal_acc_at_th={best_ba:.3f})")
    print(f"Fold {fold_i} results: train_bal_acc={train_bal_acc:.3f} | test_bal_acc={test_bal_acc:.3f} | "
          f"test_AP={test_metrics['ap'][-1]:.3f}")


# ----------------------------
# Summary
# ----------------------------
tb, ts    = mean_std(test_metrics["bal_acc"])
tap, tas  = mean_std(test_metrics["ap"])
rn, rns   = mean_std(test_metrics["recall_neg"])
rp, rps   = mean_std(test_metrics["recall_pos"])
tf1, tf1s = mean_std(test_metrics["f1"])

trb, _    = mean_std(train_metrics["bal_acc"])
trap, _   = mean_std(train_metrics["ap"])

print("\n" + "="*90)
print(f"[{sheet_name}] patient-wise nested CV + band-subset selection")
print(f"TEST : bal_acc={tb:.3f}±{ts:.3f} | AP={tap:.3f}±{tas:.3f} | "
      f"recall_neg={rn:.3f}±{rns:.3f} | recall_pos={rp:.3f}±{rps:.3f} | F1={tf1:.3f}±{tf1s:.3f}")
print(f"TRAIN: bal_acc={trb:.3f} | AP={trap:.3f}")

# Print which subsets were chosen per fold
print("\nChosen band subsets per outer fold:")
for i, ss in enumerate(chosen_subsets, 1):
    print(f"  Fold {i}: {ss}")

# Tally band frequencies
from collections import Counter
band_counts = Counter()
for ss in chosen_subsets:
    for b in ss:
        band_counts[b] += 1

print("\nBand selection frequency across outer folds:")
for b in selected_columns:
    print(f"  {b}: {band_counts[b]}/{len(chosen_subsets)} folds")


Found 108 files with labels. Building patient-level labels...
Unique patients: 107
Threshold: -2.0 | Neg: 54 Pos: 54 | AP baseline: 0.500

Loading band-wise features from sheet='Z_FFT_abs_bandpower_uV2' for bands=['Delta', 'Theta', 'Alpha', 'Beta', 'HighBeta', 'Alpha1', 'Alpha2', 'Beta1', 'Beta2', 'Beta3']
Loaded caches for 108/108 files.
Example cache keys for first sample: ['Delta', 'Theta', 'Alpha', 'Beta', 'HighBeta', 'Alpha1', 'Alpha2', 'Beta1', 'Beta2', 'Beta3']

Outer CV: 5 splits x 2 repeats = 10 folds total.
Band-subset search space: 1023 subsets (non-empty).

[Outer fold 1/10] train_samples=86 test_samples=22 | train_patients=86 test_patients=21
Searching best band subset via inner CV...
  progress: 102/1023 subsets tried | current best ('Theta', 'Alpha', 'Alpha1') score=0.622
  progress: 204/1023 subsets tried | current best ('Theta', 'Alpha', 'Alpha1') score=0.622
  progress: 306/1023 subsets tried | current best ('Theta', 'Alpha', 'Alpha1') score=0.622
  progress: 408/1023

In [2]:
Z_FFT_abs_bandpower_uV2 ['Alpha2',  'HighBeta', 'Alpha', 'Theta']  9, 6, 5,4

SyntaxError: invalid syntax (422955020.py, line 1)